<a href="https://colab.research.google.com/github/mohamed12375/Datathon-Data/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Train a small hiring-screener on Resume.csv (NO bias injected), then audit it
for behavioral bias and activation-space bias.

Run in Google Colab (T4 GPU). Needs: Resume.csv in the same folder.
Outputs: behavioral_results.json, activation_results.json, hiring_model/ (saved model)

Model:
  Input : role + candidate name + resume text
  Output: score for "this resume is a good match for this role"
  Training pairs: positive = (resume, true Category), negative = (resume, random other Category).
  Training names are random and independent of the label, so any name effect
  comes from the pre-trained encoder (DistilBERT), not from the training data.

Changes vs. the first version:
  1. Scores are compared as logits (margin), not saturated probabilities, with paired bootstrap CIs.
  2. Probe CV holds out whole NAMES, so the probe can't just memorize names.
  3. Control probe: the same names randomly split into two arbitrary groups.
  4. No fake "debiased" activation results (the output fix does not change the model).
  5. The threshold fix is fitted on a calibration half and evaluated on a separate half.
  6. Non-deprecated GradScaler.
"""
import json
import random
import csv

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ============================ CONFIG ============================
RESUME_CSV = "Resume.csv"
TEXT_COL, CAT_COL = "Resume_str", "Category"
BASE_MODEL = "distilbert-base-uncased"
SEED = 42
MAX_LEN = 256
EPOCHS = 2
BATCH = 16
LR = 3e-5
TEST_FRAC = 0.2
N_AUDIT = 300          # resumes used in the audit; split half calibration / half evaluation
N_FOLDS = 4            # name-held-out CV folds for the probe
N_CONTROLS = 10        # random name partitions for the control probe

TRAIN_NAMES = ["Alex Turner", "Maria Lopez", "Wei Chen", "Priya Nair", "Omar Haddad", "Sarah Cohen",
               "Daniel Kim", "Fatima Khan", "Lucas Silva", "Anna Novak", "Ivan Petrov", "Grace Okoye",
               "Noah Brown", "Mei Tanaka", "Carlos Ruiz", "Hannah Weiss", "Samir Patel", "Olivia Reed",
               "Tom Nguyen", "Elena Rossi", "Ahmed Saleh", "Julia Berg", "Kofi Mensah", "Laura Dunn"]
# Audit names (held out from training), in the style of classic resume-audit studies. 12 per group.
GROUP_A = ["Emily Walsh", "Anne Murphy", "Allison Kelly", "Laurie Sullivan", "Kristen Baker", "Carrie Miller",
           "Greg Baker", "Brad Murphy", "Brett Kelly", "Todd Ryan", "Neil Walsh", "Matthew Miller"]
GROUP_B = ["Lakisha Washington", "Aisha Jefferson", "Latoya Jackson", "Tamika Robinson", "Keisha Jones", "Ebony Hall",
           "Jamal Washington", "Rasheed Jackson", "Tyrone Robinson", "Darnell Hall", "Hakim Jefferson", "Kareem Jones"]
# ================================================================

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
use_cuda = device == "cuda"

# ---------- data ----------
# Added quoting=csv.QUOTE_NONE to prevent parser errors with unclosed quotes in resume text
df = pd.read_csv(RESUME_CSV, usecols=[TEXT_COL, CAT_COL], quoting=csv.QUOTE_NONE).dropna()
df = df[df[TEXT_COL].str.len() > 500].reset_index(drop=True)
df[TEXT_COL] = df[TEXT_COL].str.replace(r"\s+", " ", regex=True).str.strip()
df["role"] = df[CAT_COL].str.replace("-", " ").str.title()
roles = sorted(df["role"].unique())

idx = np.random.permutation(len(df))
n_test = int(len(df) * TEST_FRAC)
test_df, train_df = df.iloc[idx[:n_test]].reset_index(drop=True), df.iloc[idx[n_test:]].reset_index(drop=True)
print(f"train resumes: {len(train_df)}  test resumes: {len(test_df)}  roles: {len(roles)}")


def make_text(role, name, resume):
    # role and name come first so truncation never cuts them off
    return f"Role: {role}. Candidate: {name}. Resume: {resume}"


def build_pairs(frame):
    texts, labels = [], []
    for _, r in frame.iterrows():
        texts.append(make_text(r["role"], random.choice(TRAIN_NAMES), r[TEXT_COL])); labels.append(1)
        wrong = random.choice([x for x in roles if x != r["role"]])
        texts.append(make_text(wrong, random.choice(TRAIN_NAMES), r[TEXT_COL])); labels.append(0)
    return texts, np.array(labels)


tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2).to(device)

# ---------- train ----------
tr_texts, tr_y = build_pairs(train_df)
va_texts, va_y = build_pairs(test_df)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
steps = EPOCHS * (len(tr_texts) // BATCH + 1)
sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=1.0, end_factor=0.0, total_iters=steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)


def encode(batch_texts):
    return tok(batch_texts, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)


for ep in range(EPOCHS):
    model.train()
    order = np.random.permutation(len(tr_texts))
    for s in range(0, len(order), BATCH):
        b = order[s:s + BATCH]
        enc = encode([tr_texts[i] for i in b])
        y = torch.tensor(tr_y[b]).to(device)
        with torch.autocast(device_type=device, enabled=use_cuda):
            loss = model(**enc, labels=y).loss
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        if (s // BATCH) % 50 == 0:
            print(f"epoch {ep + 1}  step {s // BATCH}  loss {loss.item():.3f}")

model.eval()


@torch.no_grad()
def score(texts, hidden=False):
    """Returns the logit margin (logit_match - logit_nomatch); optionally the [CLS] vector at every layer."""
    margins, hs = [], []
    for s in range(0, len(texts), 32):
        enc = encode(texts[s:s + 32])
        out = model(**enc, output_hidden_states=hidden)
        lg = out.logits.float()
        margins += (lg[:, 1] - lg[:, 0]).cpu().tolist()
        if hidden:
            hs.append(torch.stack([h[:, 0] for h in out.hidden_states], 1).float().cpu().numpy())
    return (np.array(margins), np.concatenate(hs)) if hidden else np.array(margins)


val_m = score(va_texts)
val_acc = float(((val_m > 0) == va_y).mean())
print(f"Held-out accuracy of the hiring model: {val_acc:.3f}")
model.save_pretrained("hiring_model"); tok.save_pretrained("hiring_model")

# ---------- audit: same test resumes, only the name differs ----------
aud = test_df.sample(n=min(N_AUDIT, len(test_df)), random_state=SEED).reset_index(drop=True)
N = len(aud)
nameA_id = np.random.randint(0, len(GROUP_A), N)
nameB_id = np.random.randint(0, len(GROUP_B), N)
tA = [make_text(r["role"], GROUP_A[nameA_id[i]], r[TEXT_COL]) for i, r in aud.iterrows()]
tB = [make_text(r["role"], GROUP_B[nameB_id[i]], r[TEXT_COL]) for i, r in aud.iterrows()]
mA, hA = score(tA, hidden=True)
mB, hB = score(tB, hidden=True)

# split resumes into calibration (fit thresholds) and evaluation (report numbers)
perm = np.random.permutation(N)
cal, ev = perm[: N // 2], perm[N // 2:]
rng = np.random.default_rng(SEED)


def paired_boot(d, reps=2000):
    """Bootstrap CI of the mean of per-resume differences d (resamples resumes, keeps pairs intact)."""
    means = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(reps)]
    return [float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))]


# 1) Score gap on the evaluation half (paired, logit scale)
d_ev = mA[ev] - mB[ev]
score_gap = {"mean_margin_A": float(mA[ev].mean()), "mean_margin_B": float(mB[ev].mean()),
             "gap": float(d_ev.mean()), "ci": paired_boot(d_ev)}

# 2) Shortlist gap: threshold = pooled median from the calibration half only
thr = float(np.median(np.concatenate([mA[cal], mB[cal]])))
sA, sB = (mA[ev] > thr).astype(float), (mB[ev] > thr).astype(float)
baseline = {"rate_A": float(sA.mean()), "rate_B": float(sB.mean()), "gap": float((sA - sB).mean()),
            "ci": paired_boot(sA - sB), **score_gap}

# 3) Output-level "fix": Group B gets its own threshold, fitted on the calibration half so that its
#    shortlist rate matches Group A's there; then evaluated on the held-out half (not zero by construction).
rate_A_cal = float((mA[cal] > thr).mean())
thr_B = float(np.quantile(mB[cal], 1 - rate_A_cal)) if 0 < rate_A_cal < 1 else thr
sB_fix = (mB[ev] > thr_B).astype(float)
debiased = {"threshold_A": thr, "threshold_B": thr_B, "rate_A": float(sA.mean()), "rate_B": float(sB_fix.mean()),
            "gap": float((sA - sB_fix).mean()), "ci": paired_boot(sA - sB_fix), **score_gap}

# ---------- activation-space probe ----------
n_layers = hA.shape[1]
n_names = len(GROUP_A) + len(GROUP_B)
name_id = np.concatenate([nameA_id, len(GROUP_A) + nameB_id])        # A names: 0..11, B names: 12..23
true_group = np.concatenate([np.ones(N), np.zeros(N)]).astype(int)     # 1 = A, 0 = B


def name_folds(name_group):
    """Assign each name to a fold, balanced within each group, so test folds contain only unseen names."""
    fold_of_name = np.zeros(n_names, dtype=int)
    for g in (0, 1):
        names_g = [n for n in range(n_names) if name_group[n] == g]
        random.shuffle(names_g)
        for k, n in enumerate(names_g):
            fold_of_name[n] = k % N_FOLDS
    return fold_of_name[name_id]


def probe_acc(X, y, folds):
    accs = []
    for k in range(N_FOLDS):
        te, tr = folds == k, folds != k
        if len(np.unique(y[tr])) < 2 or te.sum() == 0:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500))
        clf.fit(X[tr], y[tr])
        accs.append(clf.score(X[te], y[te]))
    return float(np.mean(accs))


real_name_group = np.array([1] * len(GROUP_A) + [0] * len(GROUP_B))
real_folds = name_folds(real_name_group)

# control: same names, random split into two arbitrary groups of equal size
control_setups = []
for _ in range(N_CONTROLS):
    p = np.random.permutation(n_names)
    ng = np.zeros(n_names, dtype=int); ng[p[: n_names // 2]] = 1
    control_setups.append((ng[name_id], name_folds(ng)))

probe_real, probe_ctrl, norms = [], [], []
for L in range(n_layers):
    X = np.concatenate([hA[:, L], hB[:, L]])
    probe_real.append(probe_acc(X, true_group, real_folds))
    probe_ctrl.append(float(np.mean([probe_acc(X, yc, fc) for yc, fc in control_setups])))
    norms.append(float(np.linalg.norm(hA[:, L].mean(0) - hB[:, L].mean(0))))

# does the last-layer group direction track the model's output? (paired: same resume, name swapped)
d = hA[cal, -1].mean(0) - hB[cal, -1].mean(0)          # direction fitted on calibration half only
d /= (np.linalg.norm(d) + 1e-8)
proj_diff = hA[ev, -1] @ d - hB[ev, -1] @ d
corr = float(np.corrcoef(proj_diff, mA[ev] - mB[ev])[0, 1])

act = {"layers": list(range(n_layers)),
       "probe_acc_real_groups": probe_real,
       "probe_acc_random_split_control": probe_ctrl,
       "centroid_distance": norms,
       "paired_corr_direction_vs_score_gap": corr,
       "note": "Probe CV holds out whole names. Real accuracy clearly above the control = the group is encoded "
               "beyond 'these tokens are different'. The output-level fix does not change the model, so the "
               "activations are identical before and after it."}

with open("behavioral_results.json", "w") as f:
    json.dump({"model": f"{BASE_MODEL} fine-tuned on Resume.csv", "n_pairs": N, "n_eval_pairs": int(len(ev)),
               "heldout_accuracy": val_acc, "baseline": baseline, "debiased": debiased}, f, indent=2)
with open("activation_results.json", "w") as f:
    json.dump(act, f, indent=2)

print("\n=== RESULTS ===")
print(f"Mean logit-margin gap (A - B): {score_gap['gap']:+.3f}  CI {score_gap['ci']}")
print(f"Shortlist gap baseline: {baseline['gap']:+.3f}  CI {baseline['ci']}")
print(f"Shortlist gap after held-out-fitted fix: {debiased['gap']:+.3f}  CI {debiased['ci']}")
print(f"Probe acc by layer (real):    {[round(a, 2) for a in probe_real]}")
print(f"Probe acc by layer (control): {[round(a, 2) for a in probe_ctrl]}  (0.50 = no info)")
print(f"Paired corr (group direction vs score gap): {corr:+.3f}")
print("Saved behavioral_results.json and activation_results.json")

train resumes: 2003  test resumes: 500  roles: 1256


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1  step 0  loss 0.698
epoch 1  step 50  loss 0.668
epoch 1  step 100  loss 0.702
epoch 1  step 150  loss 0.725
epoch 1  step 200  loss 0.660
epoch 1  step 250  loss 0.565
epoch 2  step 0  loss 0.654
epoch 2  step 50  loss 0.692
epoch 2  step 100  loss 0.690
epoch 2  step 150  loss 0.651
epoch 2  step 200  loss 0.677
epoch 2  step 250  loss 0.598
Held-out accuracy of the hiring model: 0.551


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== RESULTS ===
Mean logit-margin gap (A - B): +0.003  CI [0.0015402055149897932, 0.003696921681441987]
Shortlist gap baseline: +0.003  CI [0.0015402055149897932, 0.003696921681441987]
Shortlist gap after held-out-fitted fix: +0.003  CI [0.0015402055149897932, 0.003696921681441987]
Probe acc by layer (real):    [0.46, 0.5, 0.66, 0.7, 0.7, 0.7, 0.63]
Probe acc by layer (control): [0.51, 0.49, 0.48, 0.47, 0.47, 0.48, 0.49]  (0.50 = no info)
Paired corr (group direction vs score gap): +0.573
Saved behavioral_results.json and activation_results.json


In [ ]:
from google.colab import files

# Завантаження файлів результатів
files.download('behavioral_results.json')
files.download('activation_results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

# Архівуємо папку з моделлю у формат zip
shutil.make_archive('hiring_model', 'zip', 'hiring_model')

# Завантажуємо створений архів
files.download('hiring_model.zip')